In [22]:
import torch
import numpy
import soundfile as sf

from wavenet.data import DATA_DIR, AudioChunks
from wavenet.model import WaveNet
from torch.utils.data import DataLoader
import torch.nn.functional as F
from pathlib import Path

device = "mps" if torch.backends.mps.is_available() else "cpu"

In [29]:
train_data = torch.load(DATA_DIR / "train_8000.pt")
val_data   = torch.load(DATA_DIR / "val_8000.pt")

In [30]:
model = WaveNet(R=64, S=128).to(device = device)
opt = torch.optim.AdamW(model.parameters(), lr = 1e-3)

In [31]:
T, B = 4096, 8
train_dl = DataLoader(AudioChunks(train_data, T, model.rf, n_items=10_000), batch_size=B)
val_dl   = DataLoader(AudioChunks(val_data, T, model.rf, random=False), batch_size=B)

In [34]:
@torch.no_grad()
def evaluate(max_batches=50):
    model.eval()
    losses = []
    for i, (x, y) in enumerate(val_dl):
        if i == max_batches:
            break
        x = x.to(device)
        y = y.to(device)
        
        logits = model(x)
        loss = F.cross_entropy(logits, y, ignore_index=-1)
        
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)

In [37]:
max_steps, eval_every = 500, 250
step, best_val = 0, float('inf')

while step < max_steps:
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        
        # forward
        logits = model(x)
        loss = F.cross_entropy(logits, y, ignore_index=-1)
        
        # backward
        opt.zero_grad()
        loss.backward()
        opt.step()
        
        if step % eval_every == 0:
            val = evaluate()
            print(f"step {step}: train {loss.item():.3f}  val {val:.3f}")
            
            if val < best_val:
                best_val = val
                torch.save({"model": model.state_dict(), "step": step, "val": val}, "../checkpoints/ckpt.pt")
        
        step += 1
        if step >= max_steps:
            break

step 0: train 5.547  val 5.529
step 250: train 3.996  val 3.669


In [38]:
loss.item()

3.3020830154418945